# Enhanced Inference Pipeline with Abstraction Metrics
Adds abstraction-aware evaluation metrics and LLM-as-Judge using the same Llama model.

**New Metrics:**
- **Abstraction Score**: Measures how much the summary differs from source (1 - n-gram overlap)
- **Compression Ratio**: Generated length / Source length
- **LLM Judge Score**: Llama evaluates quality on 1-5 scale across 4 dimensions

## 1. Setup

In [19]:
# Install deps
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub 'numpy<2.0' 'scipy>=1.10' matplotlib seaborn

!python -m spacy download it_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 72.6 MB/s  0:00:00m0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')


In [20]:
import os
import gc
import re
import torch
import json
import spacy
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

# Load spacy for sentence segmentation
nlp = spacy.load("it_core_news_sm")

## 2. Configuration

In [21]:
# Auth
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    HF_TOKEN = os.getenv("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"

login(token=HF_TOKEN)

# Simplified config - only best performing model
SIGEXT_CONFIG = {
    "model_id": "LookUpMark/sigext-wits-it-10k-060t",
    "skip_samples": 25000,
    "threshold": 0.60
}

# Use 8-bit for best performance
QUANT_CONFIG = {
    "load_in_8bit": True
}

GLOBAL_CONFIG = {
    "llm_model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "num_test_samples": 10,
    "max_length": 2048,
    "output_dir": "./results_enhanced"
}

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)
print(f"Testing with {GLOBAL_CONFIG['num_test_samples']} samples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Testing with 10 samples


## 3. Enhanced Prompts

In [22]:
# Prompt per summarization - OTTIMIZZATO PER VERA ASTRAZIONE
SUMMARY_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Sei un redattore enciclopedico professionista. Devi scrivere riassunti astrattivi in italiano.

REGOLE TASSATIVE:
1. Scrivi ESATTAMENTE UN paragrafo di prosa fluida (50-80 parole). NIENTE titoli, NIENTE elenchi puntati, NIENTE formattazione.
2. Inizia con una definizione: '[Soggetto] è...' oppure '[Soggetto] fu...'
3. SINTETIZZA le informazioni - combina più fatti in singole frasi scorrevoli.
4. NON copiare MAI frasi dal testo originale. Riscrivi tutto completamente.
5. Usa strutture sintattiche diverse dalla fonte.
6. Concentrati sull'ESSENZA, non sui dettagli.
<|eot_id|><|start_header_id|>user<|end_header_id|>
TESTO ORIGINALE:
{source}

CONCETTI CHIAVE (integrali, riformulandoli con parole tue):
{keyphrases}

Scrivi un riassunto in un SINGOLO PARAGRAFO in italiano. Ricorda: NIENTE formattazione, NIENTE titoli, solo prosa fluida.<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""


# LLM-as-Judge prompt - Blindated version with JSON safety
JUDGE_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert summary evaluator. Compare the GENERATED summary against the GOLD REFERENCE.
Respond ONLY with valid JSON.

STRICT JSON RULES:
1. Use double quotes for keys and values.
2. DO NOT use double quotes INSIDE the reason strings (use single quotes ' instead).
3. Max 15 words per reason.
<|eot_id|><|start_header_id|>user<|end_header_id|>

GOLD REFERENCE SUMMARY (Ground Truth):
{reference}

GENERATED SUMMARY (To Evaluate):
{generated}

---

Rate on 4 criteria (1-5 each) with brief justification:
1. FAITHFULNESS: Does it align factually with Reference? (1=Hallucination/Contradiction, 5=Perfectly aligned).
2. COMPLETENESS: Does it capture the main contribution and results? (1=Missing key info, 5=Comprehensive).
3. CONCISENESS: Is it fluid and efficient? (1=Verbose/Repetitive, 5=Concise).
4. ABSTRACTION: Does it rephrase concepts originally? (1=Copy-paste from source, 5=Novel synthesis).

Respond with ONLY this JSON format:
{{"faithfulness": X, "faithfulness_reason": "...", "completeness": X, "completeness_reason": "...", "conciseness": X, "conciseness_reason": "...", "abstraction": X, "abstraction_reason": "..."}}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

## 4. Helper Functions

In [23]:
def load_sigext_model(model_id):
    print(f"  Loading SigExt: {model_id}...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(model_id).to(device)
    print(f"  SigExt loaded on {device}")
    return model, tokenizer

## 5. New Abstraction Metrics

In [24]:
def get_ngrams(text, n=3):
    """Extract n-grams from text."""
    words = text.lower().split()
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]


def compute_abstraction_score(source, generated, n=3):
    """
    Compute abstraction score: 1 - (n-gram overlap with source).
    Higher = more abstractive (less copying).
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)
    
    if not gen_ngrams:
        return 1.0  # Empty summary = no copying
    
    copied = sum(1 for ng in gen_ngrams if ng in source_ngrams)
    copy_ratio = copied / len(gen_ngrams)
    
    return 1.0 - copy_ratio


def compute_compression_ratio(source, generated):
    """Compute compression ratio (lower = more compressed)."""
    if len(source) == 0:
        return 1.0
    return len(generated) / len(source)


def compute_novel_ngrams(source, generated, n=2):
    """
    Compute percentage of n-grams in generated that are NOT in source.
    Higher = more novel content.
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)
    
    if not gen_ngrams:
        return 0.0
    
    novel = sum(1 for ng in gen_ngrams if ng not in source_ngrams)
    return novel / len(gen_ngrams)

## 6. LLM-as-Judge Evaluation

In [25]:
def parse_judge_response(response_text):
    """Parse judge response with fallback for malformed JSON."""
    ENGLISH_KEYS = ['faithfulness', 'completeness', 'conciseness', 'abstraction']
    ERROR_RESPONSE = {k: 1 for k in ENGLISH_KEYS}
    ERROR_RESPONSE.update({f"{k}_reason": "Parsing Error" for k in ENGLISH_KEYS})
    
    try:
        # Try direct parsing
        return json.loads(response_text)
    except json.JSONDecodeError:
        try:
            # Fallback: Find JSON block with regex
            match = re.search(r"\{.*\}", response_text, re.DOTALL)
            if match:
                return json.loads(match.group())
        except:
            pass
    
    print(f"      JSON PARSING ERROR on: {response_text[:50]}...")
    return ERROR_RESPONSE


def llm_judge_evaluate(source, generated, reference, judge_chain):
    """
    Use the same LLM to judge the quality of the generated summary.
    Returns dict with scores and justifications for: faithfulness, completeness, conciseness, abstraction
    """
    ENGLISH_KEYS = ['faithfulness', 'completeness', 'conciseness', 'abstraction']
    DEFAULT_SCORES = {k: 3 for k in ENGLISH_KEYS}
    DEFAULT_SCORES.update({f"{k}_reason": "Unable to evaluate" for k in ENGLISH_KEYS})
    
    try:
        # Get LLM judgment - compare generated vs reference (Gold Standard approach)
        result = judge_chain.invoke({
            "generated": generated,
            "reference": reference
        })
        
        # Extract text after assistant header
        result_text = result.split("assistant<|end_header_id|>")[-1].strip()
        
        # Use robust parsing
        scores = parse_judge_response(result_text)
        
        # Validate and clamp scores
        for key in ENGLISH_KEYS:
            if key not in scores:
                scores[key] = 3
            else:
                scores[key] = max(1, min(5, int(scores[key])))
            reason_key = f"{key}_reason"
            if reason_key not in scores:
                scores[reason_key] = ""
        
        return scores
            
    except Exception as e:
        print(f"      Judge error: {e}")
        return DEFAULT_SCORES.copy()

## 7. Enhanced Evaluation Function

In [26]:
def run_enhanced_evaluation(processed_data, summary_chain, judge_chain):
    """Run evaluation with traditional + abstraction + LLM-judge metrics."""
    
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    
    metrics = {
        # Traditional
        "bert": [], "rouge": [], "kir": [],
        # Abstraction
        "abstraction": [], "compression": [], "novel_ngrams": [],
        # LLM Judge (English keys)
        "judge_faithfulness": [], "judge_completeness": [], 
        "judge_conciseness": [], "judge_abstraction": []
    }
    samples = []
    
    for item in tqdm(processed_data, desc="    Generating & Evaluating"):
        try:
            keys_text = item['keyphrases']
            salient_sents = item['salient_sentences']
            
            # Generate summary
            res = summary_chain.invoke({"source": item['source'], "keyphrases": keys_text})
            gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()
            
            # === TRADITIONAL METRICS ===
            
            # ROUGE-1
            rouge_score = scorer.score(item['reference'], gen_summary)['rouge1'].fmeasure
            metrics["rouge"].append(rouge_score)
            
            # BERTScore
            _, _, F1 = bert_score(
                [gen_summary], [item['reference']], lang="it", verbose=False
            )
            bert_sc = F1.mean().item()
            metrics["bert"].append(bert_sc)
            
            # KIR
            kir_score = 0.0
            if salient_sents:
                gen_lower = gen_summary.lower()
                hits = 0
                for sent in salient_sents:
                    words = [w.lower() for w in sent.split() if len(w) > 4]
                    if words:
                        word_hits = sum(1 for w in words if w in gen_lower)
                        if word_hits / len(words) > 0.3:
                            hits += 1
                kir_score = hits / len(salient_sents)
            metrics["kir"].append(kir_score)
            
            # === ABSTRACTION METRICS ===
            
            abstraction = compute_abstraction_score(item['source'], gen_summary)
            compression = compute_compression_ratio(item['source'], gen_summary)
            novel = compute_novel_ngrams(item['source'], gen_summary)
            
            metrics["abstraction"].append(abstraction)
            metrics["compression"].append(compression)
            metrics["novel_ngrams"].append(novel)
            
            # === LLM JUDGE ===
            
            judge_scores = llm_judge_evaluate(
                item['source'], gen_summary, item['reference'], judge_chain
            )
            
            metrics["judge_faithfulness"].append(judge_scores["faithfulness"])
            metrics["judge_completeness"].append(judge_scores["completeness"])
            metrics["judge_conciseness"].append(judge_scores["conciseness"])
            metrics["judge_abstraction"].append(judge_scores["abstraction"])
            
            # Store sample details
            samples.append({
                "source": item['source'][:500] + "..." if len(item['source']) > 500 else item['source'],
                "reference": item['reference'],
                "salient_sentences": salient_sents,
                "generated_summary": gen_summary,
                "scores": {
                    "bert": float(bert_sc),
                    "rouge": float(rouge_score),
                    "kir": float(kir_score),
                    "abstraction": float(abstraction),
                    "compression": float(compression),
                    "novel_ngrams": float(novel),
                    "judge": judge_scores
                }
            })
                
        except Exception as e:
            print(f"    Error: {e}")
            continue
    
    return metrics, samples

## 8. Main Evaluation Loop

In [27]:
print("="*60)
print("PHASE 1: LOADING MODELS")
print("="*60)

# Load SigExt
sigext_model, sigext_tokenizer = load_sigext_model(SIGEXT_CONFIG["model_id"])

# Load test data
test_data = get_test_data(
    SIGEXT_CONFIG["skip_samples"],
    GLOBAL_CONFIG["num_test_samples"]
)

# Extract salient sentences
processed_data = preprocess_dataset(
    test_data,
    sigext_model,
    sigext_tokenizer,
    GLOBAL_CONFIG["max_length"]
)

# Cleanup SigExt
del sigext_model, sigext_tokenizer
clear_gpu_memory()

print("\n" + "="*60)
print("PHASE 2: LOADING LLM")
print("="*60)

# Load LLM
llm_model, llm_tokenizer = load_llm(
    GLOBAL_CONFIG["llm_model_id"], 
    QUANT_CONFIG
)

# Create text generation pipeline
gen_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=256,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# Create chains
summary_prompt = PromptTemplate(template=SUMMARY_PROMPT, input_variables=["source", "keyphrases"])
summary_chain = summary_prompt | llm | StrOutputParser()

judge_prompt = PromptTemplate(
    template=JUDGE_PROMPT, 
    input_variables=["source_truncated", "generated", "reference"]
)
judge_chain = judge_prompt | llm | StrOutputParser()

print("\n" + "="*60)
print("PHASE 3: RUNNING EVALUATION")
print("="*60)

metrics, samples = run_enhanced_evaluation(processed_data, summary_chain, judge_chain)

print("\n" + "="*60)
print("EVALUATION COMPLETE!")
print("="*60)

PHASE 1: LOADING MODELS
  Loading SigExt: LookUpMark/sigext-wits-it-10k-060t...
  SigExt loaded on cuda
  Loading test data (skipping 25000)...


Repo card metadata block was not found. Setting CardData to empty.


  Test data ready: 10 samples


    Extracting Salient Sentences:   0%|          | 0/10 [00:00<?, ?it/s]


PHASE 2: LOADING LLM
  Loading LLM (8-bit): meta-llama/Llama-3.1-8B-Instruct...


...

Device set to use cuda:0



PHASE 3: RUNNING EVALUATION


    Generating & Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: bfcd191b-8c05-41bf-b30e-cfca3498d4b3)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: bb72d15c-c630-481c-94c0-0c9228a6b421)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json
Retrying in 1s [Retry 1/5].



EVALUATION COMPLETE!


## 9. Results Summary

In [28]:
# Compute and display results
results = {
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": SIGEXT_CONFIG["model_id"],
        "llm_model": GLOBAL_CONFIG["llm_model_id"],
        "num_samples": len(samples),
        "prompt_type": "optimized_abstractive_v2"
    },
    "metrics": {}
}

# Aggregate metrics
for m in ["bert", "rouge", "kir", "abstraction", "compression", "novel_ngrams"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

for m in ["judge_faithfulness", "judge_completeness", "judge_conciseness", "judge_abstraction"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

if metrics.get("judge_faithfulness"):
    overall = [sum(metrics[f"judge_{k}"][i] for k in ["faithfulness", "completeness", "conciseness", "abstraction"]) 
               for i in range(len(metrics["judge_faithfulness"]))]
    results["metrics"]["judge_overall"] = {"mean": float(np.mean(overall)) / 4, "std": float(np.std(overall)) / 4}

results["samples"] = samples

# Print summary
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

m = results["metrics"]
if "bert" in m:
    print(f"\nTraditional Metrics:")
    print(f"  BERT Score:  {m['bert']['mean']:.4f} +/- {m['bert']['std']:.4f}")
    print(f"  ROUGE-1:     {m['rouge']['mean']:.4f} +/- {m['rouge']['std']:.4f}")
    print(f"  KIR:         {m['kir']['mean']:.2%}")

if "abstraction" in m:
    print(f"\nAbstraction Metrics:")
    print(f"  Abstraction: {m['abstraction']['mean']:.4f}")
    print(f"  Novel:       {m['novel_ngrams']['mean']:.2%}")
    print(f"  Compression: {m['compression']['mean']:.2%}")

if "judge_faithfulness" in m:
    print(f"\nLLM Judge (1-5):")
    print(f"  Faithfulness: {m['judge_faithfulness']['mean']:.2f}")
    print(f"  Completeness: {m['judge_completeness']['mean']:.2f}")
    print(f"  Conciseness:  {m['judge_conciseness']['mean']:.2f}")
    print(f"  Abstraction:  {m['judge_abstraction']['mean']:.2f}")
    print(f"  Overall:      {m['judge_overall']['mean']:.2f}")

# Save
output_file = os.path.join(GLOBAL_CONFIG["output_dir"], "results_enhanced.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {output_file}")


RESULTS SUMMARY

Traditional Metrics:
  BERT Score:  0.6561 +/- 0.0519
  ROUGE-1:     0.1811 +/- 0.0900
  KIR:         48.02%

Abstraction Metrics:
  Abstraction: 0.8557
  Novel:       70.71%
  Compression: 25.65%

LLM Judge (1-5):
  Faithfulness: 4.20
  Completeness: 4.40
  Conciseness:  3.60
  Abstraction:  2.00
  Overall:      3.55

Saved: ./results_enhanced/results_enhanced.json


## 10. Sample Analysis

In [29]:
# Sample outputs
print("=" * 60)
print("SAMPLE OUTPUTS")
print("=" * 60)

for idx in [0, len(samples)//2, len(samples)-1]:
    s = samples[idx]
    print(f"\n[Sample {idx}]")
    print(f"Reference: {s['reference'][:200]}...")
    print(f"Generated: {s['generated_summary'][:200]}...")
    sc = s['scores']
    print(f"Scores: BERT={sc['bert']:.2f} ROUGE={sc['rouge']:.2f} KIR={sc['kir']:.2f} Abstr={sc['abstraction']:.2f}")


SAMPLE OUTPUTS

[Sample 0]
Reference: 


'''''1,039/Smoothed Out Slappy Hours''''' è una compilation della punk rock band statunitense Green Day, pubblicata nel 1991 dalla Lookout! Records.
...
Generated: Green Day è un gruppo musicale punk rock statunitense, fondato nel 1987 a Berkeley, California. Il loro primo album, 1,039/Smoothed Out Slappy Hours, è una raccolta di pezzi del loro primo album 39/Sm...
Scores: BERT=0.71 ROUGE=0.30 KIR=0.46 Abstr=0.80

[Sample 5]
Reference: 

La sua opera più nota è ''Il Novellino'', una raccolta di 50 racconti, ciascuno preceduto da una lettera di dedica a un personaggio famoso e con un epilogo contenente la "morale" della storia. I rac...
Generated: Masuccio Salernitano è un autore italiano del Rinascimento, figlio di Margaritella de' Mariconda e Luigi Guardati, membro della nobile famiglia dei Guardati. Nato probabilmente tra marzo e aprile, si ...
Scores: BERT=0.72 ROUGE=0.34 KIR=0.43 Abstr=0.77

[Sample 9]
Reference: 

I '''Sicani'''  erano un p

## 11. Cleanup

In [30]:
# Cleanup
del llm_model, llm_tokenizer, gen_pipe, llm
clear_gpu_memory()

print(" Cleanup complete!")

 Cleanup complete!
